In [1]:
import tensorflow as tf
import numpy as np
import os

from tensorflow.python.framework.tensor_spec import TensorSpec

from Training.TensorSlider import WindowSlider

print(os.getcwd())
tfrecordpath = "../Data/tfrecords/"


2025-02-01 01:16:32.742360: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-01 01:16:32.746411: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-01 01:16:32.757074: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738368992.777895   45705 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738368992.782692   45705 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-01 01:16:32.801082: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

/mnt/c/Users/Alexander/PycharmProjects/levbot/Training


### Load in data
#### Define schema


In [2]:
def decode(record_bytes):
    # Function for parsing each record in the tf files
    example = tf.io.parse_single_example(
        # Data
        record_bytes,

        # Schema
        {
        'Timeframe': tf.io.FixedLenFeature([], tf.string),
        'timestamp': tf.io.RaggedFeature(dtype=tf.int64),
        'Open': tf.io.RaggedFeature(dtype=tf.float32),
        'High': tf.io.RaggedFeature(dtype=tf.float32),
        'Low': tf.io.RaggedFeature(dtype=tf.float32),
        'Close': tf.io.RaggedFeature(dtype=tf.float32),
        'Volume': tf.io.RaggedFeature(dtype=tf.float32),
        }
        )

    return example

In [3]:
path = tfrecordpath + "BTCUSD_PERP/1m.tfrecord"

dataset1m = tf.data.TFRecordDataset(path,  num_parallel_reads = tf.data.AUTOTUNE)
dataset1m = dataset1m.map(decode, num_parallel_calls = tf.data.AUTOTUNE)

path = tfrecordpath + "BTCUSD_PERP/5m.tfrecord"

dataset5m = tf.data.TFRecordDataset(path,  num_parallel_reads = tf.data.AUTOTUNE)
dataset5m = dataset5m.map(decode, num_parallel_calls = tf.data.AUTOTUNE)



W0000 00:00:1738369001.406272   45705 gpu_device.cc:2344] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
for data in dataset1m:
    print(data["timestamp"][0])

2025-02-01 01:16:41.642286: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144


tf.Tensor(1597132139, shape=(), dtype=int64)


2025-02-01 01:16:43.067593: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [5]:
import TensorSlider

In [6]:
WS = TensorSlider.WindowSlider(30, 10, {"1m": dataset1m})
WS2 = TensorSlider.WindowSlider(30, 10, {"1m": dataset1m, "5m": dataset5m})

In [7]:
for time in WS:
    print(time)
    break
for time in WS2:
    print(time)
    break

(<tf.Tensor: shape=(1, 5, 30), dtype=float64, numpy=
array([[[11769.5       , 11762.90039062, 11762.5       , 11762.        ,
         11763.20019531, 11763.20019531, 11763.20019531, 11757.59960938,
         11758.        , 11752.20019531, 11751.40039062, 11752.20019531,
         11777.79980469, 11778.09960938, 11773.90039062, 11761.90039062,
         11775.5       , 11780.        , 11777.59960938, 11776.        ,
         11770.20019531, 11769.5       , 11764.40039062, 11759.40039062,
         11766.70019531, 11771.90039062, 11749.90039062, 11760.59960938,
         11773.5       , 11773.09960938],
        [11769.5       , 11779.70019531, 11771.20019531, 11767.79980469,
         11763.20019531, 11763.20019531, 11763.20019531, 11759.70019531,
         11758.        , 11752.20019531, 11751.40039062, 11753.29980469,
         11780.20019531, 11778.09960938, 11773.90039062, 11774.5       ,
         11775.5       , 11780.        , 11777.90039062, 11776.40039062,
         11770.20019531, 1176

In [8]:
iterate = WS

for i in range(200):
    next = iterate.__next__()
print(next[0].shape)
print(next[1].shape)

(1, 5, 30)
(11, 5)


In [9]:
def slider():
    return TensorSlider.WindowSlider(30, 10,{"1m": dataset1m, "5m": dataset5m})

dataset = tf.data.Dataset.from_generator(slider, 
    output_signature=(
        tf.TensorSpec((2,5,30), dtype=tf.float32),
        tf.TensorSpec((11,5), dtype=tf.float32))
    )

In [10]:
def createLabels(data, lookforward):
    """
    create input labels from the lookaehad data
    """
    
    ohlc = lookforward
    ohlc /= ohlc[0,:] # divide by open price
    ohlc -= 1 # zero out
    ohlc *= 100 # convert to percentage
    
    high = ohlc[tf.math.argmin(ohlc[:,1]), 1]
    low = ohlc[tf.math.argmin(ohlc[:,2]), 2]
    
    return data, [low, high]
    
    
    

In [11]:
import time
t0 = None
for i, data in enumerate(dataset.map(createLabels)):
    if i == 0:
        t0 = time.time()
    if i > 10:
        tdelta = time.time() - t0
        print(f"Time taken for {i+1} calls: {tdelta}s, thats {((tdelta/(i+1))*1000):.2f}ms per call")
        print(data[1])
        break
        

Time taken for 12 calls: 0.057442665100097656s, thats 4.79ms per call
tf.Tensor([-0.17485023 -0.07759333], shape=(2,), dtype=float32)
